# 02 — Model Training

Produces the artifacts consumed by `03_model_evaluation.ipynb`:
- `models/xgboost_final_model.pkl` (point-estimate model)
- `models/xgboost_model_safety.pkl` (quantile / safety-bound model)
- `data/processed/model_results_final.csv` (test-set predictions, with `unit`/`cycle` preserved, plus the baseline's predictions for statistical comparison)
- `data/processed/test_features_engineered.csv` (test-set feature matrix, for SHAP)
- `data/processed/baseline_metrics.json` (baseline metrics, for comparison in 03)
- `data/processed/feature_reference_ranges.json` (per-feature training ranges, used as an out-of-distribution guard at inference time)
- `data/processed/run_info.json` (MLflow run IDs, so 03 can log evaluation artifacts into the same run instead of a disconnected one)

All parameters (RUL cap, rolling window, business costs, thresholds...) come from
`configs/config.yaml` — no hardcoded magic numbers.

The Optuna objective factory (`make_objective`) is imported from `src/train.py`
rather than redefined here, so this notebook and the non-interactive training
script (`python src/train.py`, what `dvc.yaml` actually runs) can never drift
on what "a good hyperparameter search" means.

In [12]:
import sys
import os
import json
import yaml
import joblib
import numpy as np
import pandas as pd
import mlflow
import mlflow.xgboost

sys.path.append(os.path.abspath('../src'))
from data_loader import clean_and_save_data
from features import add_temporal_features, get_sensor_columns
from evaluation import nasa_score, pinball_loss, compute_feature_reference_ranges
from train import make_objective

with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

mlflow.set_tracking_uri(f"sqlite:///{os.path.abspath(os.path.join('..', config['mlflow']['tracking_db']))}")
mlflow.set_experiment(config['mlflow']['experiment_name'])

print("Config loaded:")
print(json.dumps(config, indent=2))

Config loaded:
{
  "data": {
    "raw_train_path": "data/raw/train_FD001.txt",
    "cleaned_train_path": "data/processed/train_cleaned.csv",
    "max_rul": 125
  },
  "features": {
    "window_size": 10
  },
  "split": {
    "test_size": 0.2,
    "random_state": 42
  },
  "optuna": {
    "n_trials": 25,
    "n_estimators_cap": 2000,
    "early_stopping_rounds": 30,
    "cv_folds": 5
  },
  "safety_model": {
    "quantile_alpha": 0.1
  },
  "business": {
    "cost_unplanned_failure": 50000,
    "cost_preventive_maintenance": 10000,
    "cost_false_positive": 2000,
    "critical_threshold_cycles": 5,
    "alert_threshold_cycles": 10
  },
  "health_score": {
    "inspect_threshold": 50,
    "ground_threshold": 20
  },
  "evaluation": {
    "rul_slice_bins": [
      [
        0,
        20
      ],
      [
        20,
        50
      ],
      [
        50,
        100
      ],
      [
        100,
        null
      ]
    ],
    "bootstrap_n_resamples": 1000,
    "bootstrap_ci_level": 0.9

## ETL & Data Preprocessing (Piecewise RUL Capping)

In [13]:
clean_and_save_data(
    input_path=f"../{config['data']['raw_train_path']}",
    output_path=f"../{config['data']['cleaned_train_path']}",
    max_rul=config['data']['max_rul']
)

 Lecture de : ../data/raw/train_FD001.txt
 Fichier nettoyé sauvegardé dans : ../data/processed/train_cleaned.csv


## Data Splitting

In [14]:
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv(f"../{config['data']['cleaned_train_path']}")

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=config['split']['test_size'],
    random_state=config['split']['random_state']
)
train_idx, test_idx = next(gss.split(df, groups=df['unit']))

train_data = df.iloc[train_idx].reset_index(drop=True).copy()
test_data = df.iloc[test_idx].reset_index(drop=True).copy()

X_train = train_data.drop(['unit', 'RUL'], axis=1)
y_train = train_data['RUL']
X_test = test_data.drop(['unit', 'RUL'], axis=1)
y_test = test_data['RUL']

print(f"Step 1 OK: {test_data['unit'].nunique()} engines in Test set.")
print(f"Test data columns: {test_data.columns.tolist()}")

Step 1 OK: 20 engines in Test set.
Test data columns: ['unit', 'cycle', 'os1', 'os2', 'os3', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21', 'RUL']


## Feature Engineering

In [15]:
sensor_cols = get_sensor_columns(train_data, exclude=['unit', 'cycle', 'RUL'])
window_size = config['features']['window_size']

train_data_eng = add_temporal_features(train_data, sensor_cols, window_size=window_size)
test_data_eng = add_temporal_features(test_data, sensor_cols, window_size=window_size)

X_train = train_data_eng.drop(['unit', 'RUL'], axis=1).drop(columns=['cycle'], errors='ignore')
X_test = test_data_eng.drop(['unit', 'RUL'], axis=1).drop(columns=['cycle'], errors='ignore')

print("--- FEATURE ENGINEERING SUMMARY ---")
print(f"Initial features count : {len(sensor_cols) + 1}")
print(f"Final features count   : {X_train.shape[1]}")
print(f"New columns preview    : {[c for c in X_train.columns if 'roll' in c][:5]}")

--- FEATURE ENGINEERING SUMMARY ---
Initial features count : 25
Final features count   : 72
New columns preview    : ['os1_roll_mean', 'os1_roll_std', 'os2_roll_mean', 'os2_roll_std', 'os3_roll_mean']


## Baseline Model (Linear Regression)

In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_baseline = baseline.predict(X_test)

rmse_baseline = np.sqrt(mean_squared_error(y_test, y_baseline))
mae_baseline = mean_absolute_error(y_test, y_baseline)
nasa_baseline = nasa_score(y_test, y_baseline)

print("--- BASELINE RESULTS (LINEAR REGRESSION) ---")
print(f"Baseline RMSE : {rmse_baseline:.2f} cycles")
print(f"Baseline MAE  : {mae_baseline:.2f} cycles")
print(f"Baseline NASA : {nasa_baseline:.0f}")

# Exported so 03 can compare without retraining the baseline
baseline_metrics = {
    "rmse_baseline": float(rmse_baseline),
    "mae_baseline": float(mae_baseline),
    "nasa_baseline": float(nasa_baseline)
}
with open('../data/processed/baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=2)

--- BASELINE RESULTS (LINEAR REGRESSION) ---
Baseline RMSE : 19.78 cycles
Baseline MAE  : 15.86 cycles
Baseline NASA : 26644


## Hyperparameter Optimization (Optuna) — Point-Estimate Model

In [17]:
import optuna
import xgboost as xgb
from sklearn.model_selection import GroupKFold, GroupShuffleSplit

study = optuna.create_study(direction='minimize')
study.optimize(
    make_objective(X_train, y_train, train_data['unit'], config),
    n_trials=config['optuna']['n_trials']
)

print("--- OPTUNA OPTIMIZATION COMPLETE (objective = NASA Score) ---")
print(f"Best CV NASA Score: {study.best_value:.0f}")
print(f"Best Parameters: {study.best_params}")

[I 2026-07-23 00:18:33,345] A new study created in memory with name: no-name-4cdf7f32-dbc1-4a54-803e-497dd6baa0c5
[I 2026-07-23 00:18:42,426] Trial 0 finished with value: 53867.1737437533 and parameters: {'max_depth': 7, 'learning_rate': 0.014744832340062571, 'subsample': 0.8201780069445305, 'colsample_bytree': 0.8530046581573587}. Best is trial 0 with value: 53867.1737437533.
[I 2026-07-23 00:18:44,401] Trial 1 finished with value: 48666.75462604688 and parameters: {'max_depth': 3, 'learning_rate': 0.025098321177666763, 'subsample': 0.6712024103235404, 'colsample_bytree': 0.6015328148870525}. Best is trial 1 with value: 48666.75462604688.
[I 2026-07-23 00:18:48,356] Trial 2 finished with value: 48014.49914614336 and parameters: {'max_depth': 3, 'learning_rate': 0.010968944422021844, 'subsample': 0.6199490909717414, 'colsample_bytree': 0.7771268312773296}. Best is trial 2 with value: 48014.49914614336.
[I 2026-07-23 00:18:52,191] Trial 3 finished with value: 53107.87927732071 and param

--- OPTUNA OPTIMIZATION COMPLETE (objective = NASA Score) ---
Best CV NASA Score: 48014
Best Parameters: {'max_depth': 3, 'learning_rate': 0.010968944422021844, 'subsample': 0.6199490909717414, 'colsample_bytree': 0.7771268312773296}


## Final Model Training

In [18]:
from sklearn.metrics import r2_score

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=config['split']['random_state'])
tr_idx, val_idx = next(gss_val.split(X_train, y_train, groups=train_data['unit']))
X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

final_model = xgb.XGBRegressor(
    **study.best_params,
    n_estimators=config['optuna']['n_estimators_cap'],
    random_state=config['split']['random_state'],
    early_stopping_rounds=config['optuna']['early_stopping_rounds']
)
final_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

print(f"Rounds actually used : {final_model.best_iteration}")

y_pred = final_model.predict(X_test)
rmse_final = np.sqrt(mean_squared_error(y_test, y_pred))
r2_final = r2_score(y_test, y_pred)
nasa_final = nasa_score(y_test, y_pred)

print(f"Final NASA Score   : {nasa_final:.0f}")
print(f"NASA improvement   : {((nasa_baseline - nasa_final) / nasa_baseline * 100):.1f}%")
print(f"Final RMSE         : {rmse_final:.2f} cycles")
print(f"R2 Score           : {r2_final:.4f}")

joblib.dump(final_model, f"../{config['models']['final_model_path']}")

# run_id is captured so 03_model_evaluation.ipynb can resume logging into THIS
# exact run later (mlflow.start_run(run_id=...)) instead of creating a
# disconnected "evaluation" run — evaluation artifacts stay attached to the
# model version that produced them.
with mlflow.start_run(run_name="xgboost_final_model") as run:
    champion_run_id = run.info.run_id
    mlflow.log_params(study.best_params)
    mlflow.log_param("n_estimators_cap", config['optuna']['n_estimators_cap'])
    mlflow.log_param("early_stopping_rounds", config['optuna']['early_stopping_rounds'])
    mlflow.log_metric("cv_nasa_score", study.best_value)
    mlflow.log_metric("test_rmse", rmse_final)
    mlflow.log_metric("test_r2", r2_final)
    mlflow.log_metric("test_nasa_score", nasa_final)
    mlflow.log_metric("baseline_rmse", rmse_baseline)
    mlflow.log_metric("nasa_improvement_pct", (nasa_baseline - nasa_final) / nasa_baseline * 100)
    mlflow.xgboost.log_model(final_model, name="model")

print(f"MLflow champion run id: {champion_run_id}")

Rounds actually used : 455
Final NASA Score   : 34137
NASA improvement   : -28.1%
Final RMSE         : 16.57 cycles
R2 Score           : 0.8422
MLflow champion run id: 4051ca22e55141d28a9909abf2cecbe8


## Robust Validation Strategy (Group K-Fold stability check)

In [19]:
gkf = GroupKFold(n_splits=config['optuna']['cv_folds'])
cv_rmse_scores = []

print("--- STARTING GROUP-BASED CROSS-VALIDATION ---")

for i, (tr_idx_cv, val_idx_cv) in enumerate(gkf.split(X_train, y_train, groups=train_data['unit'])):
    X_tr_fold, X_val_fold = X_train.iloc[tr_idx_cv], X_train.iloc[val_idx_cv]
    y_tr_fold, y_val_fold = y_train.iloc[tr_idx_cv], y_train.iloc[val_idx_cv]

    model_cv = xgb.XGBRegressor(
        **study.best_params,
        n_estimators=config['optuna']['n_estimators_cap'],
        random_state=config['split']['random_state'],
        early_stopping_rounds=config['optuna']['early_stopping_rounds']
    )
    model_cv.fit(X_tr_fold, y_tr_fold, eval_set=[(X_val_fold, y_val_fold)], verbose=False)

    fold_preds = model_cv.predict(X_val_fold)
    fold_rmse = np.sqrt(mean_squared_error(y_val_fold, fold_preds))
    cv_rmse_scores.append(fold_rmse)
    print(f"Fold {i+1}: RMSE = {fold_rmse:.2f} cycles")

mean_cv = np.mean(cv_rmse_scores)
std_cv = np.std(cv_rmse_scores)

print("---------------------------------------------")
print(f"Final Mean CV RMSE  : {mean_cv:.2f} cycles")
print(f"Final Std Deviation : {std_cv:.2f} cycles")
print("---------------------------------------------")

if std_cv < 5:
    print("STATUS: Model is STABLE across engine groups.")
else:
    print("STATUS: High variance detected - investigate engine outliers.")

--- STARTING GROUP-BASED CROSS-VALIDATION ---
Fold 1: RMSE = 21.50 cycles
Fold 2: RMSE = 17.44 cycles
Fold 3: RMSE = 18.97 cycles
Fold 4: RMSE = 19.15 cycles
Fold 5: RMSE = 17.79 cycles
---------------------------------------------
Final Mean CV RMSE  : 18.97 cycles
Final Std Deviation : 1.43 cycles
---------------------------------------------
STATUS: Model is STABLE across engine groups.


## Safety & Uncertainty Layer (Quantile Regression)

In [20]:
alpha = config['safety_model']['quantile_alpha']

study_safety = optuna.create_study(direction='minimize')
study_safety.optimize(
    make_objective(
        X_train, y_train, train_data['unit'], config,
        extra_params={'objective': 'reg:quantileerror', 'quantile_alpha': alpha},
        score_fn=lambda yt, yp: pinball_loss(yt, yp, alpha=alpha)
    ),
    n_trials=config['optuna']['n_trials']
)
print(f"Best Pinball Loss (safety) : {study_safety.best_value:.3f}")
print(f"Best Parameters (safety)   : {study_safety.best_params}")

model_safety = xgb.XGBRegressor(
    **study_safety.best_params,
    n_estimators=config['optuna']['n_estimators_cap'],
    random_state=config['split']['random_state'],
    early_stopping_rounds=config['optuna']['early_stopping_rounds']
)
model_safety.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

safe_limit = np.maximum(model_safety.predict(X_test), 0)

mean_pred = y_pred.mean()
mean_safe = safe_limit.mean()
print(f"Champion Average RUL   : {mean_pred:.2f} cycles")
print(f"Safety Average RUL     : {mean_safe:.2f} cycles")
print(f"Average Safety Buffer  : {(mean_pred - mean_safe):.1f} cycles")
print("VALIDATION SUCCESS" if mean_safe < mean_pred else "WARNING: Safety limit too high")

joblib.dump(model_safety, f"../{config['models']['safety_model_path']}")

with mlflow.start_run(run_name="xgboost_safety_model") as run:
    safety_run_id = run.info.run_id
    mlflow.log_params(study_safety.best_params)
    mlflow.log_param("quantile_alpha", alpha)
    mlflow.log_metric("cv_pinball_loss", study_safety.best_value)
    mlflow.log_metric("mean_safety_rul", float(mean_safe))
    mlflow.log_metric("mean_safety_buffer", float(mean_pred - mean_safe))
    mlflow.xgboost.log_model(model_safety, name="model")

print(f"MLflow safety run id: {safety_run_id}")

[I 2026-07-23 00:19:54,608] A new study created in memory with name: no-name-8a84ab5e-b4d6-4140-a36c-250a78d56216
[I 2026-07-23 00:20:01,425] Trial 0 finished with value: 3.571791101257259 and parameters: {'max_depth': 5, 'learning_rate': 0.022959725072741948, 'subsample': 0.8938453575176651, 'colsample_bytree': 0.6772639913669943}. Best is trial 0 with value: 3.571791101257259.
[I 2026-07-23 00:20:04,780] Trial 1 finished with value: 3.554434456470679 and parameters: {'max_depth': 5, 'learning_rate': 0.057309190591166605, 'subsample': 0.6174062003995504, 'colsample_bytree': 0.8671422334686449}. Best is trial 1 with value: 3.554434456470679.
[I 2026-07-23 00:20:18,245] Trial 2 finished with value: 3.534291441795162 and parameters: {'max_depth': 4, 'learning_rate': 0.008082764760501899, 'subsample': 0.6902503370383654, 'colsample_bytree': 0.8798967507361988}. Best is trial 2 with value: 3.534291441795162.
[I 2026-07-23 00:20:40,389] Trial 3 finished with value: 3.596403118834354 and par

Best Pinball Loss (safety) : 3.529
Best Parameters (safety)   : {'max_depth': 5, 'learning_rate': 0.00815360148932532, 'subsample': 0.7133920715959925, 'colsample_bytree': 0.6022305786719273}
Champion Average RUL   : 86.08 cycles
Safety Average RUL     : 86.26 cycles
Average Safety Buffer  : -0.2 cycles
MLflow safety run id: 2d646970814b430e9a2c1b193dcd604a


## Export Artifacts for Evaluation (03)

This notebook writes **only** persisted artifacts (models + CSV + JSON).
`03_model_evaluation.ipynb` reloads everything from disk and depends on no
in-memory variable from this kernel — it can run on its own, in the order
`dvc.yaml` enforces (`train` before `evaluate`).

Files exported:
- `model_results_final.csv` — identifiers (`unit`, `cycle`) + target + champion
  predictions + **baseline predictions** (so 03 can run a paired statistical
  comparison, not just compare two aggregate numbers) + safety bound.
- `test_features_engineered.csv` — the feature matrix itself, needed for SHAP.
- `feature_reference_ranges.json` — per-feature training ranges, the
  out-of-distribution guard used by `predict_with_explanation`.
- `run_info.json` — MLflow run IDs, so 03 logs evaluation artifacts into the
  same run as the model that produced them.

In [21]:
results_export = test_data_eng[['unit', 'cycle']].reset_index(drop=True).copy()
results_export['RUL'] = y_test.reset_index(drop=True).values
results_export['predicted_RUL'] = y_pred
results_export['baseline_predicted_RUL'] = y_baseline
results_export['safety_RUL'] = safe_limit
results_export['absolute_error'] = (results_export['RUL'] - results_export['predicted_RUL']).abs()
results_export.to_csv('../data/processed/model_results_final.csv', index=False)

features_export = test_data_eng.drop(columns=['RUL']).reset_index(drop=True).copy()
features_export.to_csv('../data/processed/test_features_engineered.csv', index=False)

reference_ranges = compute_feature_reference_ranges(
    X_train,
    lower_q=config['evaluation']['ood_lower_quantile'],
    upper_q=config['evaluation']['ood_upper_quantile']
)
with open('../data/processed/feature_reference_ranges.json', 'w') as f:
    json.dump(reference_ranges, f, indent=2)

run_info = {"champion_run_id": champion_run_id, "safety_run_id": safety_run_id}
with open('../data/processed/run_info.json', 'w') as f:
    json.dump(run_info, f, indent=2)

print("Artifacts exported:")
print(" - data/processed/model_results_final.csv          ", results_export.shape)
print(" - data/processed/test_features_engineered.csv     ", features_export.shape)
print(" - data/processed/feature_reference_ranges.json    ", f"({len(reference_ranges)} features)")
print(" - data/processed/run_info.json                    ", run_info)

Artifacts exported:
 - data/processed/model_results_final.csv           (4070, 7)
 - data/processed/test_features_engineered.csv      (4070, 74)
 - data/processed/feature_reference_ranges.json     (72 features)
 - data/processed/run_info.json                     {'champion_run_id': '4051ca22e55141d28a9909abf2cecbe8', 'safety_run_id': '2d646970814b430e9a2c1b193dcd604a'}


## Production Inference — Sanity Check

In [22]:
from inference import predict_unit_health, predict_with_explanation
from evaluation import get_shap_explainer

test_id = test_data['unit'].iloc[0]

# predict_unit_health recomputes rolling features itself, so it takes raw
# (non feature-engineered) test_data, not test_data_eng.
prediction = predict_unit_health(test_data, test_id, final_model, window_size=window_size)

if prediction is not None:
    print(f"Sanity Check - Unit {test_id} RUL: {prediction:.2f} cycles")
    print("System Status: Operational and Robust.")
else:
    print(f"Sanity Check - Unit {test_id}: prediction failed, check pipeline.")

# Operator-facing version: same prediction, plus an out-of-distribution flag
# and the top local SHAP drivers behind this specific number.
explainer = get_shap_explainer(final_model)
explained = predict_with_explanation(
    test_data, test_id, final_model, explainer, reference_ranges,
    window_size=window_size
)
if explained is not None:
    print("\n--- Explained prediction ---")
    print(f"Unit {explained['unit_id']}: {explained['predicted_rul']:.2f} cycles")
    print(f"Out of distribution: {explained['out_of_distribution']} "
          f"(violation ratio: {explained['ood_violation_ratio']})")
    print("Top drivers:")
    for r in explained['top_reasons']:
        direction = "shortens" if r['shap_impact'] < 0 else "extends"
        print(f"  - {r['feature']}: {direction} RUL (impact {r['shap_impact']:+.2f})")

Sanity Check - Unit 1 RUL: 5.78 cycles
System Status: Operational and Robust.

--- Explained prediction ---
Unit 1: 5.78 cycles
Out of distribution: True (violation ratio: 0.208)
Top drivers:
  - s4_roll_mean: shortens RUL (impact -17.55)
  - s11_roll_mean: shortens RUL (impact -13.03)
  - s2_roll_mean: shortens RUL (impact -12.10)
